# 05 · Bộ dữ liệu SFT v2 (T07b)
Làm sạch, giá qua công cụ SQL, RAFT, mẫu khuôn (đặt bàn nhiều lượt, từ chối, khiếu nại…). Dataset card: `data/splits_v2/README.md`.

In [1]:
import sys, os, json, pathlib, pandas as pd
# tìm thư mục gốc repo (chứa code/src/kimochi) rồi làm việc từ đó — notebook đặt ở đâu cũng chạy được
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'code/src/kimochi').exists())
os.chdir(ROOT); sys.path.insert(0, str(ROOT / 'code/src'))
pd.set_option('display.max_colwidth', 80)

In [2]:
from kimochi import prepare_sft_v2
s = prepare_sft_v2.main()
{k: v for k, v in s.items() if k not in ('format_by_split', 'intent_by_split')}

{'split_sizes': {'train': 4980, 'val': 580},
 'cleaning_log': {'drop_exact_duplicate': 60,
  'drop_conflicting_question': 239,
  'price_stripped': 839,
  'drop_price_inconsistent': 7,
  'drop_price_unstrippable': 33},
 'source': {'kimochi_v1': 3991,
  'kimochi_v1+raft': 1077,
  'template_v2': 257,
  'kimochi_v1+db': 235},
 'multi_turn': 433,
 'assistant_turns_total': 6547,
 'system_prompt_variants': 3}

In [3]:
pd.DataFrame(s['format_by_split']).fillna(0).astype(int)

,train,val
chitchat,16,1
closed_book,3587,404
handoff,13,1
oos_code,6,0
oos_competitor,6,0
oos_general,7,1
oos_homework,5,1
oos_injection,9,3
oos_medical,4,0
raft_neg,213,28


In [4]:
pd.DataFrame(s['intent_by_split']).fillna(0).astype(int)

,train,val
allergy_diet,576,65
chitchat,16,1
complaint,31,4
menu_info,2858,326
out_of_scope,37,5
policy_promo,177,16
price_query,342,45
recommendation,677,78
reservation_create,168,25
store_info,98,15


In [5]:
rows = pd.read_json('data/splits_v2/train.jsonl', lines=True)
for fmt in ['tool_price', 'raft_neg', 'reservation_mt']:
    r = rows[rows.meta.map(lambda m: m['format']) == fmt].iloc[0]
    print('=====', fmt)
    for m in r.messages[1:]:
        print(f"  [{m['role']}] {m['content'][:200]}")

===== tool_price
  [user] Salad Gà Đào giá bao nhiêu?
  [assistant] <tool_call>{"name": "get_menu_item", "arguments": {"item": "Salad Gà Đào"}}</tool_call>
  [tool] {"found": true, "item": "Salad Gà Đào", "category": "Salad", "price_vnd": 109000, "allergens": [], "allergen_verified": false, "available": true}
  [assistant] Dạ một phần Salad Gà Đào có giá 109.000 VNĐ nha bạn, khẩu phần vừa đủ cho 1 người ăn đó ạ. 💸
===== raft_neg
  [user] Ngữ cảnh:
[doc_menu_M067#c0] Rong Nho Sốt Mè (Size bé): Dạ Rong Nho Sốt Mè (Size bé) là món khai vị siêu hấp dẫn của tiệm nhà Chi đóiu. Rong nho tươi có vị hơi mặn nhẹ, khi dùng kèm sốt mè thì thơm bé
  [assistant] Dạ câu này em chưa đủ thông tin để trả lời chắc chắn về Salad Rong Biển ạ. Khách iu gọi hotline 0879096969 (Bình Thạnh) / 0888912344 (Gò Vấp) hoặc để em chuyển nhân viên nha.
===== reservation_mt
  [user] mình muốn đặt bàn 2 người ở Bình Thạnh
  [assistant] Dạ khách iu dự định đến lúc mấy giờ, ngày nào ạ?
  [user] 11h ngay 18/11
  [assistant

In [6]:
from kimochi import llm
from pathlib import Path
print('Số mẫu huấn luyện sau khi tách theo lượt assistant:', len(llm.sft_dataset(ROOT, 'train')))

Số mẫu huấn luyện sau khi tách theo lượt assistant: 5843
